# 01 Data Inspection

Purpose of this notebook:

- Load the raw loan dataset.
- Inspect target distribution and modelling population.
- Review all variables for data type, missingness, cardinality and basic data-quality issues.
- Produce reusable inspection artefacts in `data/processed/01.data_inspection/` as both `.pkl` and `.xlsx`.

## 1. Imports and configuration

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


RAW_CANDIDATES = [
    Path("../data/raw/loan_processed_data.csv"),
    Path("data/raw/loan_processed_data.csv"),
]

RAW_DATA_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find loan_processed_data.csv. Expected it under data/raw/ or /mnt/data/."
    )

PROCESSED_DIR = Path("../data/processed/01.data_inspection")
if not PROCESSED_DIR.parent.exists():
    PROCESSED_DIR = Path("data/processed/01.data_inspection")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = "loan_status"
BAD_LABEL = "Charged Off"
GOOD_LABEL = "Fully Paid"

print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Processed output directory: {PROCESSED_DIR.resolve()}")

## 2. Load raw data

In [ ]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
df.head()

## 3. Dataset overview

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

constant_cols = [col for col in df.columns if df[col].nunique(dropna=False) <= 1]
all_missing_cols = [col for col in df.columns if df[col].isna().all()]

n_rows, n_cols = df.shape
n_duplicates = int(df.duplicated().sum())

data_overview = pd.DataFrame({
    "metric": [
        "n_rows",
        "n_columns",
        "n_numeric_columns",
        "n_categorical_columns",
        "n_duplicate_rows",
        "duplicate_rows_pct",
        "n_constant_columns",
        "n_all_missing_columns",
        "target_column",
        "bad_label",
        "good_label",
    ],
    "value": [
        n_rows,
        n_cols,
        len(numeric_cols),
        len(categorical_cols),
        n_duplicates,
        n_duplicates / n_rows if n_rows else np.nan,
        len(constant_cols),
        len(all_missing_cols),
        TARGET_COL,
        BAD_LABEL,
        GOOD_LABEL,
    ]
})

data_overview

## 4. Target variable inspection

In [ ]:
if TARGET_COL not in df.columns:
    raise KeyError(f"Target column '{TARGET_COL}' was not found in the dataset.")

target_distribution = (
    df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis("target_class")
    .reset_index(name="count")
)
target_distribution["pct"] = target_distribution["count"] / len(df)
target_distribution["is_bad"] = target_distribution["target_class"].eq(BAD_LABEL)

bad_rate = df[TARGET_COL].eq(BAD_LABEL).mean()
print(f"Observed bad rate ({BAD_LABEL}): {bad_rate:.2%}")

target_distribution

## 5. Column-level data inspection

This table is the main high-level inventory for **all variables**.

In [ ]:
def infer_semantic_type(series: pd.Series) -> str:
    """Lightweight semantic classification for inspection purposes only."""
    name = series.name.lower()
    dtype = series.dtype
    non_missing = series.dropna()
    
    if series.name == TARGET_COL:
        return "target"
    if pd.api.types.is_numeric_dtype(dtype):
        return "numeric"
    if any(token in name for token in ["_d", "date", "issue", "earliest", "last_"]):
        # Object columns such as issue_d and earliest_cr_line are often date-like in LendingClub-style data.
        parsed = pd.to_datetime(non_missing.astype(str), errors="coerce")
        if len(non_missing) > 0 and parsed.notna().mean() >= 0.70:
            return "date_like"
    return "categorical"

column_summary_rows = []
for col in df.columns:
    s = df[col]
    non_missing = s.dropna()
    unique_cnt = s.nunique(dropna=True)
    unique_cnt_incl_missing = s.nunique(dropna=False)
    missing_cnt = int(s.isna().sum())
    missing_pct = missing_cnt / len(df)
    zero_cnt = int((s == 0).sum()) if pd.api.types.is_numeric_dtype(s) else np.nan
    zero_pct = zero_cnt / len(df) if pd.api.types.is_numeric_dtype(s) else np.nan
    
    top_value = np.nan
    top_count = np.nan
    top_pct = np.nan
    if len(non_missing) > 0:
        vc = non_missing.value_counts(dropna=False)
        top_value = vc.index[0]
        top_count = int(vc.iloc[0])
        top_pct = top_count / len(df)
    
    column_summary_rows.append({
        "variable": col,
        "dtype": str(s.dtype),
        "semantic_type": infer_semantic_type(s),
        "missing_cnt": missing_cnt,
        "missing_pct": missing_pct,
        "non_missing_cnt": int(s.notna().sum()),
        "unique_cnt": int(unique_cnt),
        "unique_cnt_incl_missing": int(unique_cnt_incl_missing),
        "unique_pct": unique_cnt / len(df),
        "zero_cnt": zero_cnt,
        "zero_pct": zero_pct,
        "top_value": top_value,
        "top_count": top_count,
        "top_pct": top_pct,
        "is_constant": unique_cnt_incl_missing <= 1,
        "is_all_missing": missing_cnt == len(df),
        "is_high_cardinality": unique_cnt > 100,
    })

column_summary = pd.DataFrame(column_summary_rows)
column_summary.sort_values(["semantic_type", "missing_pct", "unique_cnt"], ascending=[True, False, False]).head(20)

## 6. Missing values review

In [ ]:
missing_values = (
    column_summary[["variable", "dtype", "semantic_type", "missing_cnt", "missing_pct", "unique_cnt"]]
    .sort_values("missing_pct", ascending=False)
    .reset_index(drop=True)
)

missing_values.head(30)

## 7. Numeric variable inspection

The table below gives distribution statistics for every numeric variable.

In [ ]:
percentiles = [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00]

numeric_summary_rows = []
for col in numeric_cols:
    s = df[col]
    q = s.quantile(percentiles).to_dict()
    numeric_summary_rows.append({
        "variable": col,
        "count": int(s.notna().sum()),
        "missing_cnt": int(s.isna().sum()),
        "missing_pct": s.isna().mean(),
        "mean": s.mean(),
        "std": s.std(),
        "min": q.get(0.00),
        "p01": q.get(0.01),
        "p05": q.get(0.05),
        "p25": q.get(0.25),
        "p50": q.get(0.50),
        "p75": q.get(0.75),
        "p95": q.get(0.95),
        "p99": q.get(0.99),
        "max": q.get(1.00),
        "skew": s.skew(),
        "kurtosis": s.kurtosis(),
        "zero_cnt": int((s == 0).sum()),
        "zero_pct": (s == 0).mean(),
        "negative_cnt": int((s < 0).sum()),
        "negative_pct": (s < 0).mean(),
    })

numeric_summary = pd.DataFrame(numeric_summary_rows)
numeric_summary.sort_values("missing_pct", ascending=False).head(20)

## 8. Outlier screening

This is a mechanical screening table. It does **not** automatically mean the values should be capped or removed.

In [ ]:
outlier_summary = numeric_summary.copy()
outlier_summary["p99_to_p50_ratio"] = np.where(
    outlier_summary["p50"].abs() > 1e-9,
    outlier_summary["p99"] / outlier_summary["p50"].replace(0, np.nan),
    np.nan,
)
outlier_summary["max_to_p99_ratio"] = np.where(
    outlier_summary["p99"].abs() > 1e-9,
    outlier_summary["max"] / outlier_summary["p99"].replace(0, np.nan),
    np.nan,
)
outlier_summary["potential_extreme_tail"] = (
    (outlier_summary["max_to_p99_ratio"].abs() >= 5) |
    (outlier_summary["p99_to_p50_ratio"].abs() >= 10)
)

outlier_summary = outlier_summary[[
    "variable", "min", "p01", "p05", "p50", "p95", "p99", "max",
    "p99_to_p50_ratio", "max_to_p99_ratio", "potential_extreme_tail",
    "missing_pct", "zero_pct", "negative_pct"
]].sort_values(["potential_extreme_tail", "max_to_p99_ratio"], ascending=[False, False])

outlier_summary.head(30)

## 9. Categorical variable inspection

For categorical variables, the notebook stores a compact distribution table with the top categories and an aggregated `__OTHER__` bucket when needed.

In [ ]:
MAX_LEVELS_PER_CATEGORICAL = 50

categorical_summary_rows = []
category_level_rows = []

for col in categorical_cols:
    s = df[col]
    vc = s.value_counts(dropna=False)
    missing_cnt = int(s.isna().sum())
    unique_cnt = int(s.nunique(dropna=True))
    
    categorical_summary_rows.append({
        "variable": col,
        "count": int(s.notna().sum()),
        "missing_cnt": missing_cnt,
        "missing_pct": missing_cnt / len(df),
        "unique_cnt": unique_cnt,
        "is_high_cardinality": unique_cnt > 100,
        "top_level": vc.index[0] if len(vc) else np.nan,
        "top_level_cnt": int(vc.iloc[0]) if len(vc) else 0,
        "top_level_pct": vc.iloc[0] / len(df) if len(vc) else np.nan,
    })
    
    for rank, (level, count) in enumerate(vc.head(MAX_LEVELS_PER_CATEGORICAL).items(), start=1):
        category_level_rows.append({
            "variable": col,
            "rank": rank,
            "level": level,
            "count": int(count),
            "pct": count / len(df),
            "is_other_bucket": False,
        })
    
    if len(vc) > MAX_LEVELS_PER_CATEGORICAL:
        other_count = int(vc.iloc[MAX_LEVELS_PER_CATEGORICAL:].sum())
        category_level_rows.append({
            "variable": col,
            "rank": MAX_LEVELS_PER_CATEGORICAL + 1,
            "level": "__OTHER__",
            "count": other_count,
            "pct": other_count / len(df),
            "is_other_bucket": True,
        })

categorical_summary = pd.DataFrame(categorical_summary_rows)
category_levels = pd.DataFrame(category_level_rows)

categorical_summary.sort_values("unique_cnt", ascending=False).head(20)

## 10. Date-like variable inspection

In [ ]:
date_candidate_cols = [
    col for col in df.columns
    if any(token in col.lower() for token in ["_d", "date", "issue", "earliest", "last_"])
]

date_summary_rows = []
parsed_dates = {}

for col in date_candidate_cols:
    parsed = pd.to_datetime(df[col], errors="coerce")
    parsed_dates[col] = parsed
    valid_pct = parsed.notna().mean()
    if valid_pct > 0:
        min_date = parsed.min()
        max_date = parsed.max()
    else:
        min_date = pd.NaT
        max_date = pd.NaT
    date_summary_rows.append({
        "variable": col,
        "dtype": str(df[col].dtype),
        "missing_pct_original": df[col].isna().mean(),
        "parseable_date_pct": valid_pct,
        "min_date": min_date,
        "max_date": max_date,
        "n_unique_parsed_dates": int(parsed.nunique(dropna=True)),
    })

date_summary = pd.DataFrame(date_summary_rows).sort_values("parseable_date_pct", ascending=False)
date_summary

## 11. Issue-date vintage profile

If `issue_d` is available and parseable, this table supports the later time-based train / validation / OOT split.

In [ ]:
if "issue_d" in df.columns:
    issue_d_parsed = pd.to_datetime(df["issue_d"], errors="coerce")
    issue_vintage = (
        pd.DataFrame({
            "issue_month": issue_d_parsed.dt.to_period("M").astype(str),
            "target": df[TARGET_COL]
        })
        .groupby("issue_month", dropna=False)
        .agg(
            observations=("target", "size"),
            bads=("target", lambda x: x.eq(BAD_LABEL).sum()),
            goods=("target", lambda x: x.eq(GOOD_LABEL).sum()),
        )
        .reset_index()
    )
    issue_vintage["bad_rate"] = issue_vintage["bads"] / issue_vintage["observations"]
else:
    issue_vintage = pd.DataFrame(columns=["issue_month", "observations", "bads", "goods", "bad_rate"])

issue_vintage.head(), issue_vintage.tail()

## 12. Target bad rate by variable levels / bands

This creates first-pass univariate target diagnostics for all non-target variables:

- Categorical variables: bad rate by top levels.
- Numeric variables: bad rate by quantile-based bins.

This is **not** final WOE/IV binning; it is only an initial risk-direction inspection.

In [ ]:
# Binary target for inspection.
df_inspect = df.copy()
df_inspect["target_bad"] = df_inspect[TARGET_COL].eq(BAD_LABEL).astype(int)

bad_rate_by_category_rows = []
for col in categorical_cols:
    if col == TARGET_COL:
        continue
    tmp = df_inspect[[col, "target_bad"]].copy()
    tmp[col] = tmp[col].astype("object").where(tmp[col].notna(), "__MISSING__")
    vc = tmp[col].value_counts(dropna=False)
    keep_levels = vc.head(MAX_LEVELS_PER_CATEGORICAL).index
    tmp["level_grouped"] = np.where(tmp[col].isin(keep_levels), tmp[col].astype(str), "__OTHER__")
    grouped = (
        tmp.groupby("level_grouped", dropna=False)
        .agg(observations=("target_bad", "size"), bads=("target_bad", "sum"), bad_rate=("target_bad", "mean"))
        .reset_index()
        .rename(columns={"level_grouped": "level"})
    )
    grouped["variable"] = col
    grouped["share"] = grouped["observations"] / len(df)
    grouped = grouped[["variable", "level", "observations", "share", "bads", "bad_rate"]]
    bad_rate_by_category_rows.append(grouped)

bad_rate_by_category = (
    pd.concat(bad_rate_by_category_rows, ignore_index=True)
    if bad_rate_by_category_rows else
    pd.DataFrame(columns=["variable", "level", "observations", "share", "bads", "bad_rate"])
)

bad_rate_by_category.head(20)

In [ ]:
MAX_NUMERIC_BINS = 10
bad_rate_by_numeric_bin_rows = []

for col in numeric_cols:
    if col == TARGET_COL:
        continue
    s = df_inspect[col]
    tmp = df_inspect[[col, "target_bad"]].copy()
    
    # Separate missing values to make their default rate visible.
    tmp["bin"] = "__MISSING__"
    non_missing_mask = tmp[col].notna()
    non_missing = tmp.loc[non_missing_mask, col]
    
    if non_missing.nunique() <= 1:
        tmp.loc[non_missing_mask, "bin"] = "single_value"
    else:
        try:
            binned = pd.qcut(non_missing, q=min(MAX_NUMERIC_BINS, non_missing.nunique()), duplicates="drop")
            tmp.loc[non_missing_mask, "bin"] = binned.astype(str)
        except ValueError:
            tmp.loc[non_missing_mask, "bin"] = "binning_failed"
    
    grouped = (
        tmp.groupby("bin", dropna=False)
        .agg(
            observations=("target_bad", "size"),
            bads=("target_bad", "sum"),
            bad_rate=("target_bad", "mean"),
            min_value=(col, "min"),
            max_value=(col, "max"),
        )
        .reset_index()
    )
    grouped["variable"] = col
    grouped["share"] = grouped["observations"] / len(df)
    grouped = grouped[["variable", "bin", "min_value", "max_value", "observations", "share", "bads", "bad_rate"]]
    bad_rate_by_numeric_bin_rows.append(grouped)

bad_rate_by_numeric_bin = (
    pd.concat(bad_rate_by_numeric_bin_rows, ignore_index=True)
    if bad_rate_by_numeric_bin_rows else
    pd.DataFrame(columns=["variable", "bin", "min_value", "max_value", "observations", "share", "bads", "bad_rate"])
)

bad_rate_by_numeric_bin.head(20)

## 13. Potential leakage candidate review

This table is only a first-pass flagging mechanism. Final exclusion decisions are made in `02_target_definition_and_leakage.ipynb`.

In [ ]:
LEAKAGE_KEYWORDS = [
    "pymnt", "payment", "recover", "collection", "settlement", "hardship",
    "last_fico", "last_credit_pull", "last_pymnt", "total_rec", "out_prncp",
    "debt_settlement", "chargeoff", "charged_off"
]

leakage_rows = []
for col in df.columns:
    lower = col.lower()
    matched_keywords = [kw for kw in LEAKAGE_KEYWORDS if kw in lower]
    if matched_keywords or col == TARGET_COL:
        leakage_rows.append({
            "variable": col,
            "matched_keywords": ", ".join(matched_keywords),
            "initial_flag": col != TARGET_COL,
            "comment": (
                "Target variable" if col == TARGET_COL else
                "Potential post-origination / post-outcome information. Review before modelling."
            )
        })

leakage_candidates = pd.DataFrame(leakage_rows)
leakage_candidates

## 14. Data quality flags summary

In [ ]:
data_quality_flags = column_summary[[
    "variable", "dtype", "semantic_type", "missing_pct", "unique_cnt",
    "is_constant", "is_all_missing", "is_high_cardinality"
]].copy()

data_quality_flags["high_missing_gt_50pct"] = data_quality_flags["missing_pct"] > 0.50
data_quality_flags["moderate_missing_10_50pct"] = data_quality_flags["missing_pct"].between(0.10, 0.50, inclusive="right")

data_quality_flags = data_quality_flags.merge(
    leakage_candidates[["variable", "initial_flag"]].rename(columns={"initial_flag": "potential_leakage_initial_flag"}),
    on="variable",
    how="left"
)
data_quality_flags["potential_leakage_initial_flag"] = data_quality_flags["potential_leakage_initial_flag"].fillna(False)

data_quality_flags.sort_values(
    ["potential_leakage_initial_flag", "high_missing_gt_50pct", "is_high_cardinality", "missing_pct"],
    ascending=[False, False, False, False]
).head(30)

## 15. Save inspection artefacts

The notebook saves:

- `data_inspection_artifacts.pkl`: a Python dictionary with all inspection outputs.
- `data_inspection_report.xlsx`: an Excel workbook with one sheet per key output table.

In [ ]:
inspection_artifacts = {
    "data_overview": data_overview,
    "target_distribution": target_distribution,
    "column_summary": column_summary,
    "missing_values": missing_values,
    "numeric_summary": numeric_summary,
    "outlier_summary": outlier_summary,
    "categorical_summary": categorical_summary,
    "category_levels_top": category_levels,
    "date_summary": date_summary,
    "issue_vintage": issue_vintage,
    "bad_rate_by_category": bad_rate_by_category,
    "bad_rate_by_numeric_bin": bad_rate_by_numeric_bin,
    "leakage_candidates": leakage_candidates,
    "data_quality_flags": data_quality_flags,
    "constant_columns": constant_cols,
    "all_missing_columns": all_missing_cols,
    "numeric_columns": numeric_cols,
    "categorical_columns": categorical_cols,
    "config": {
        "raw_data_path": str(RAW_DATA_PATH),
        "target_col": TARGET_COL,
        "bad_label": BAD_LABEL,
        "good_label": GOOD_LABEL,
        "max_levels_per_categorical": MAX_LEVELS_PER_CATEGORICAL,
        "max_numeric_bins": MAX_NUMERIC_BINS,
    }
}

pkl_path = PROCESSED_DIR / "data_inspection_artifacts.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(inspection_artifacts, f)

xlsx_path = PROCESSED_DIR / "data_inspection_report.xlsx"

sheets_to_export = {
    "data_overview": data_overview,
    "target_distribution": target_distribution,
    "column_summary": column_summary,
    "missing_values": missing_values,
    "numeric_summary": numeric_summary,
    "outlier_summary": outlier_summary,
    "categorical_summary": categorical_summary,
    "category_levels_top": category_levels,
    "date_summary": date_summary,
    "issue_vintage": issue_vintage,
    "bad_rate_category": bad_rate_by_category,
    "bad_rate_numeric": bad_rate_by_numeric_bin,
    "leakage_candidates": leakage_candidates,
    "data_quality_flags": data_quality_flags,
}

with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
    for sheet_name, table in sheets_to_export.items():
        # Excel sheet names are limited to 31 chars.
        safe_sheet_name = sheet_name[:31]
        table.to_excel(writer, sheet_name=safe_sheet_name, index=False)
        worksheet = writer.sheets[safe_sheet_name]
        
        # Basic readable formatting.
        worksheet.freeze_panes(1, 0)
        worksheet.autofilter(0, 0, max(len(table), 1), max(len(table.columns) - 1, 0))
        for idx, col_name in enumerate(table.columns):
            sample_values = table[col_name].astype(str).head(500).tolist() if len(table) else []
            width = max([len(str(col_name))] + [len(v) for v in sample_values]) + 2
            worksheet.set_column(idx, idx, min(max(width, 10), 40))

print(f"Saved pickle: {pkl_path}")
print(f"Saved Excel report: {xlsx_path}")

## 16. Notebook conclusion

Use the generated artefacts to support Notebook 02:

- Confirm target definition and good/bad population.
- Review flagged leakage candidates.
- Decide which variables should be excluded before modelling.
- Confirm whether `issue_d` supports a time-based train / validation / OOT split.

In [ ]:
print("Inspection completed successfully.")
print(f"Bad rate: {bad_rate:.2%}")
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Potential leakage candidates flagged: {len(leakage_candidates) - 1 if TARGET_COL in leakage_candidates['variable'].values else len(leakage_candidates)}")
print(f"Outputs saved under: {PROCESSED_DIR}")

## Target variable definition 
Target variable is derived directly from the final loan outcome.

Bad (1):
    Charged Off

Good (0):
    Fully Paid

The dataset contains only resolved loans and does not provide an explicit
observation window or performance window. Therefore the model predicts
historical loan outcome rather than a regulatory 12-month PD.